# Anxiety Analysis and Prediction

In [67]:
# Importing Libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder



df = pd.read_csv('anxiety.data.csv')

print(df.info())


print(df.isnull().sum())

df.dropna(axis=0, inplace = True)

print(df.isnull().sum())

df.drop('Unnamed: 0',axis=1,inplace = True)

print(df.isnull().sum())

print(df.head())






#Converting the data by LabelEncoder
Le = LabelEncoder()
target = Le.fit_transform(df.status)
print(df.status.unique())

group = df.groupby('status')



anxiety = df[df["status"]=="Anxiety"]

normal = df[df.status=="Normal"].iloc[:6159,:]

print(normal.shape)

print(anxiety.shape)

newdf = pd.concat([anxiety,normal])

newdf.reset_index(inplace = True)

print(newdf.head())

print(newdf.shape)

newdf.drop("index",inplace=True,axis=1)

print(newdf)

# EDA

newdf.status.value_counts().plot.pie(autopct="%1.2f%%",shadow=True,explode=[0.05,0.01])
plt.title("Distribution of target")
# plt.savefig("targetDistribution.png")
plt.show()

x = newdf["statement"]
y=newdf.status

# Vectorization using tf-idf

from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer()
vectorized = vectorizer.fit_transform(x)

# for i in vectorized:
#     print(i)

le = LabelEncoder()
y = le.fit_transform(y)

print(le.classes_)

dct = {}
dct[0] = le.inverse_transform([0])[0]
dct[1] = le.inverse_transform([1])[0]

print(dct)

from sklearn.naive_bayes import MultinomialNB,BernoulliNB
from sklearn.metrics import confusion_matrix,accuracy_score,f1_score
from sklearn.model_selection import train_test_split

xtrain,xtest,ytrain,ytest = train_test_split(vectorized,y,test_size=0.2,random_state=1)



nb = MultinomialNB()
nb.fit(xtrain,ytrain)

print("accuracy score",accuracy_score(ytest,nb.predict(xtest)))

print("f1 score",f1_score(ytest,nb.predict(xtest)))

cm = confusion_matrix(ytest,nb.predict(xtest))

print(cm)

sns.heatmap(cm,cmap="Blues",annot=True)
plt.show()

from sklearn.model_selection import GridSearchCV

params ={"alpha":[0.1,0.5,1.0,1.5,2.0]}

cv = GridSearchCV(nb,param_grid=params)
cv.fit(vectorized,y)
print("accuracy score",accuracy_score(ytest,cv.predict(xtest)))

alpha =cv.best_params_['alpha']

print('beat score',cv.best_score_)

nb = MultinomialNB(alpha=alpha)
nb.fit(vectorized,y)

print("accuracy:",accuracy_score(ytest,nb.predict(xtest)))

print('f1 score:',f1_score(ytest,nb.predict(xtest)))

cm = confusion_matrix(ytest,nb.predict(xtest))

sns.heatmap(cm,annot=True,cmap="Reds")
plt.show()
def pipeline(x):
    global nb,vectorizer
    vec = vectorizer.transform([x])
    value = nb.predict(vec)
    print("return:",dct[value[0]])

pipeline("i am feeling low.")





<class 'pandas.core.frame.DataFrame'>
RangeIndex: 53043 entries, 0 to 53042
Data columns (total 3 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   Unnamed: 0  53043 non-null  int64 
 1   statement   52681 non-null  object
 2   status      53043 non-null  object
dtypes: int64(1), object(2)
memory usage: 1.2+ MB
None
Unnamed: 0      0
statement     362
status          0
dtype: int64
Unnamed: 0    0
statement     0
status        0
dtype: int64
statement    0
status       0
dtype: int64
                                           statement   status
0                                         oh my gosh  Anxiety
1  trouble sleeping, confused mind, restless hear...  Anxiety
2  All wrong, back off dear, forward doubt. Stay ...  Anxiety
3  I've shifted my focus to something else but I'...  Anxiety
4  I'm restless and restless, it's been a month n...  Anxiety
['Anxiety' 'Normal' 'Depression' 'Suicidal' 'Stress' 'Bipolar'
 'Personality disorder']
(6159,

NameError: name 'printanxiety' is not defined